In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("loan_approval_data.csv")

In [ ]:
df.head()

In [ ]:
df.info()
df.isnull().sum()
df.describe()

In [ ]:
#Handle missing values of missing numerical value and categorical value
#for numerical value use mean value of the whole data
#for categorical fill the value of majority 

categorical_cols = df.select_dtypes(include=["object"]).columns
numerical_cols = df.select_dtypes(include=["number"]).columns


In [ ]:
categorical_cols
numerical_cols

In [ ]:
categorical_cols.size + numerical_cols.size

In [ ]:
from sklearn.impute import SimpleImputer

num_imp = SimpleImputer(strategy="mean")
df[numerical_cols] = num_imp.fit_transform(df[numerical_cols])

In [ ]:
df.head()

In [ ]:
cat_imp = SimpleImputer(strategy="most_frequent")
df[categorical_cols] = cat_imp.fit_transform(df[categorical_cols])

In [ ]:
df.isnull().sum()

In [ ]:
#Exploratory data analysis(EDA)
#how balanced our classes are?

classes_count = df["Loan_Approved"].value_counts()
plt.pie(classes_count,labels=["No","Yes"],autopct="%1.1f%%")
plt.title("Is Loan approved or not?")

In [ ]:
#analyse category
gender_count = df["Gender"].value_counts()
ax=sns.barplot(gender_count)
ax.bar_label(ax.containers[0])

In [ ]:
educat_count = df["Education_Level"].value_counts()
ax=sns.barplot(educat_count)
ax.bar_label(ax.containers[0])

In [ ]:
loan_purpo_count = df["Loan_Purpose"].value_counts()
ax = sns.barplot(loan_purpo_count)
ax.bar_label(ax.containers[0])

In [ ]:
property_area_count = df["Property_Area"].value_counts()
ax = sns.barplot(property_area_count)
ax.bar_label(ax.containers[0])

In [ ]:
employer_count = df["Employer_Category"].value_counts()
ax = sns.barplot(employer_count)
ax.bar_label(ax.containers[0])

In [ ]:
#analyse income
sns.histplot(
    data = df,
    x= "Applicant_Income",
    bins=20
)

In [ ]:
sns.histplot(
    data = df,
    x = "Coapplicant_Income",
    bins=20
)

In [ ]:
#outliers -box plots
sns.boxplot(
    data=df,
    x = "Loan_Approved",
    y= "Applicant_Income"
)

In [ ]:
fig,axes = plt.subplots(2,2, figsize= (12,10))
sns.boxplot(ax=axes[0,0],data=df,x= "Loan_Approved", y = "Applicant_Income")
sns.boxplot(ax=axes[0,1],data=df,x= "Loan_Approved", y = "Coapplicant_Income")
sns.boxplot(ax=axes[1,0],data=df,x= "Loan_Approved", y = "Age")
sns.boxplot(ax=axes[1,1],data=df,x= "Loan_Approved", y = "Dependents")

In [ ]:
fig,axes = plt.subplots(2,2, figsize=(12,10))
sns.boxplot(ax=axes[0,0],data= df,x ="Loan_Approved",y ="Credit_Score")
sns.boxplot(ax=axes[0,1],data= df,x ="Loan_Approved",y ="Existing_Loans")
sns.boxplot(ax=axes[1,0],data= df,x ="Loan_Approved",y ="DTI_Ratio")
sns.boxplot(ax=axes[1,1],data= df,x ="Loan_Approved",y ="Collateral_Value")

In [ ]:
fig,axes = plt.subplots(2,2,figsize=(12,10))
sns.boxplot(ax=axes[0,0],data=df,x="Loan_Approved",y= "Collateral_Value")
sns.boxplot(ax=axes[0,1],data=df,x="Loan_Approved",y= "Loan_Amount")
sns.boxplot(ax=axes[1,0],data= df,x= "Loan_Approved",y="Loan_Term")

In [ ]:
df.head()

In [ ]:
#remove applicant id because it is not contributing factor  for loan approval or not

df = df.drop("Applicant_ID",axis=1)


In [ ]:
df.head()

Encoding


In [ ]:
df


In [ ]:

from sklearn.preprocessing import LabelEncoder,OneHotEncoder

le = LabelEncoder()
df['Gender']= le.fit_transform(df['Gender'])
df['Education_Level'] = le.fit_transform(df['Education_Level'])
df['Loan_Approved'] = le.fit_transform(df['Loan_Approved'])

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
cols = ['Employment_Status','Marital_Status','Loan_Purpose', 'Property_Area','Employer_Category']
ohe = OneHotEncoder(drop="first",sparse_output=False,handle_unknown="ignore")

encoded = ohe.fit_transform(df[cols])
#convert 2d array of encoded into dataframe

encoded_df = pd.DataFrame(encoded,columns=ohe.get_feature_names_out(cols),index =df.index)

df = pd.concat([df.drop(columns=cols),encoded_df],axis=1)

In [ ]:
encoded

In [ ]:
ohe.get_feature_names_out(cols)

In [ ]:
encoded_df.head()

In [ ]:
df.head()
df.info()

Correlation Heatmap


In [ ]:
nums_cols = df.select_dtypes(include='number')
corr_matrix = nums_cols.corr()

plt.figure(figsize=(15,8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",cmap="coolwarm"
)

In [ ]:
corr_matrix

In [ ]:
nums_cols.corr()["Loan_Approved"].sort_values(ascending = False)

Train-Test-Split + Feature Scaling


In [ ]:
X = df.drop("Loan_Approved",axis=1)
y = df["Loan_Approved"]

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state = 42)

In [ ]:
X_test.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_scaled

In [ ]:
X_test_scaled

Train & Evaluate Models

In [ ]:
#logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score

log_model = LogisticRegression()
log_model.fit(X_train_scaled,y_train)

y_pred = log_model.predict(X_test_scaled)

#evaluation
print("Logistic Regression Model: ")
print("Precision: ",precision_score(y_test,y_pred))
print("Recall: ",recall_score(y_test,y_pred))
print("F1 score: ",f1_score(y_test,y_pred))
print("Accuracy: ",accuracy_score(y_test,y_pred))
print("Confusion Matrix: ",confusion_matrix(y_test,y_pred))


In [ ]:
#KNN model

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled,y_train)

y_pred = knn_model.predict(X_test_scaled)

#evaluation
print("KNN Model: ")
print("Precision: ",precision_score(y_test,y_pred))
print("Recall: ",recall_score(y_test,y_pred))
print("F1 score: ",f1_score(y_test,y_pred))
print("Accuracy: ",accuracy_score(y_test,y_pred))
print("Confusion Matrix: ",confusion_matrix(y_test,y_pred))



In [ ]:
#Naive Bayes

from sklearn.naive_bayes import GaussianNB

naive_model = GaussianNB()
naive_model.fit(X_train_scaled,y_train)

y_pred = naive_model.predict(X_test_scaled)

#evaluation
print("Naive Bayes Model: ")
print("Precision: ",precision_score(y_test,y_pred))
print("Recall: ",recall_score(y_test,y_pred))
print("F1 score: ",f1_score(y_test,y_pred))
print("Accuracy: ",accuracy_score(y_test,y_pred))
print("Confusion Matrix: ",confusion_matrix(y_test,y_pred))


#Best model on the basis of precision is Naive Bayes

Feature Engineering

In [ ]:
#Add or Transform features

df["DTI_Ratio_sq"] = df["DTI_Ratio"]**2
df["Credit_Score_sq"] = df["Credit_Score"]**2

# df["Applicant_Income_log"] =np.log1p(df["Applicant_Income"])

X= df.drop(columns=["Loan_Approved","Credit_Score","DTI_Ratio"])
y = df["Loan_Approved"]

#train test split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

#Scaling 
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train.head()

In [ ]:
#logistic Regression

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score

log_model = LogisticRegression()
log_model.fit(X_train_scaled,y_train)

y_pred = log_model.predict(X_test_scaled)

#evaluation
print("Logistic Regression Model: ")
print("Precision: ",precision_score(y_test,y_pred))
print("Recall: ",recall_score(y_test,y_pred))
print("F1 score: ",f1_score(y_test,y_pred))
print("Accuracy: ",accuracy_score(y_test,y_pred))
print("Confusion Matrix: ",confusion_matrix(y_test,y_pred))


In [ ]:
#KNN model

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix,accuracy_score,precision_score,recall_score,f1_score

knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train_scaled,y_train)

y_pred = knn_model.predict(X_test_scaled)

#evaluation
print("KNN Model: ")
print("Precision: ",precision_score(y_test,y_pred))
print("Recall: ",recall_score(y_test,y_pred))
print("F1 score: ",f1_score(y_test,y_pred))
print("Accuracy: ",accuracy_score(y_test,y_pred))
print("Confusion Matrix: ",confusion_matrix(y_test,y_pred))



In [ ]:
#Naive Bayes

from sklearn.naive_bayes import GaussianNB

naive_model = GaussianNB()
naive_model.fit(X_train_scaled,y_train)

y_pred = naive_model.predict(X_test_scaled)

#evaluation
print("Naive Bayes Model: ")
print("Precision: ",precision_score(y_test,y_pred))
print("Recall: ",recall_score(y_test,y_pred))
print("F1 score: ",f1_score(y_test,y_pred))
print("Accuracy: ",accuracy_score(y_test,y_pred))
print("Confusion Matrix: ",confusion_matrix(y_test,y_pred))
